# DistilBERT PII Detection

## Goal

Fine-tune DistilBERT for binary classification of privacy-sensitive prompts.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

Evaluation metrics match the classical ML baseline: precision, recall, F1, and confusion matrix per class.

In [ ]:
# Cell 2 -- Imports
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset

In [ ]:
# Cell 3 -- confirm GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Cell 4 -- load cleaned frozen splits generated by data_split.py
train = pd.read_parquet('../../data_splits/train.parquet')
val = pd.read_parquet('../../data_splits/val.parquet')
test = pd.read_parquet('../../data_splits/test.parquet')

# sanity check
print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)
print("\nLabel balance (train):")
print(train['label'].value_counts(normalize=True) * 100)

## Tokenization

DistilBERT requires text to be tokenized using its own tokenizer. We use `DistilBertTokenizerFast` with truncation and padding to a max length of 128 tokens, which covers the majority of examples in this dataset given the average prompt length from EDA (~157 chars / ~20 words for sensitive examples).

In [ ]:
# Cell 6 -- load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# tokenize all three splits
def tokenize(texts, labels, max_length=128):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    return encodings, list(labels)

train_encodings, train_labels = tokenize(train['text'], train['label'])
val_encodings, val_labels = tokenize(val['text'], val['label'])
test_encodings, test_labels = tokenize(test['text'], test['label'])

print("Tokenization complete.")

In [ ]:
# Cell 7 -- custom dataset class for DistilBERT
class PIIDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = PIIDataset(train_encodings, train_labels)
val_dataset = PIIDataset(val_encodings, val_labels)
test_dataset = PIIDataset(test_encodings, test_labels)

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"Test: {len(test_dataset)} examples")

## Model

We load DistilBERT with a binary classification head (`num_labels=2`). The model is fine-tuned on the training split and evaluated against the val split during training.

In [ ]:
# Cell 9 -- load DistilBERT with binary classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.to(device)
print("Model loaded and moved to:", device)

In [ ]:
# Cell 10 -- define metrics function for Trainer
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, target_names=['Safe', 'PII'], output_dict=True)
    return {
        'f1_pii': report['PII']['f1-score'],
        'f1_safe': report['Safe']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'accuracy': report['accuracy']
    }

In [ ]:
# Cell 11 -- configure training arguments
training_args = TrainingArguments(
    output_dir='../../results/distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='../../results/distilbert/logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_pii',
    fp16=True,
    seed=42
)

In [ ]:
# Cell 12 -- initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized.")

## Training

Fine-tune DistilBERT on the training split. Evaluation runs after each epoch against the validation split. Best model checkpoint is loaded at the end based on PII F1 score, since false negatives (sensitive prompts missed) are the primary concern for this project.

In [ ]:
# Cell 14 -- fine-tune DistilBERT
trainer.train()

## Evaluation

Evaluate the best checkpoint against the validation set. Metrics match the classical ML baseline: precision, recall, F1 per class, and confusion matrix. Special attention on PII recall -- false negatives represent sensitive prompts incorrectly classified as safe.

In [ ]:
# Cell 16 -- evaluate best model on validation set
val_predictions = trainer.predict(val_dataset)
val_preds = val_predictions.predictions.argmax(-1)

print(classification_report(val_labels, val_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(val_labels, val_preds))

## Test Set Evaluation

Run on the held-out test set only after validation results are satisfactory. Do not use test results to tune the model.

In [ ]:
# Cell 18 -- evaluate best model on held-out test set
test_predictions = trainer.predict(test_dataset)
test_preds = test_predictions.predictions.argmax(-1)

print(classification_report(test_labels, test_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(test_labels, test_preds))